In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
import time, pandas as pd
import datetime
options = Options()
options.add_argument("--headless=new")

df_data={'Full Name': [],
         'Regular Rating':[],
         'Blitz Rating':[],
         'Improvement': []}
browser = webdriver.Chrome(options=options)

def get_rating(id):
    global df_data, browser
    browser.get(f'https://ratings.uschess.org/player/{id}')
    time.sleep(1)
    rating_tables = browser.find_elements(By.CSS_SELECTOR, '.flex.items-center.gap-1.text-lg')
    reg_rating = rating_tables[0].text.split('/')[0].strip()
    blitz_rating = rating_tables[2].text.split('/')[0].strip()
    first_name = browser.find_elements(By.CSS_SELECTOR, '.font-regular.overflow-hidden.text-ellipsis.whitespace-nowrap.min-w-0.shrink')[1].text.upper()
    last_name = browser.find_elements(By.CSS_SELECTOR, '.font-semibold.shrink-0')[1].text
    
    button = browser.find_element(By.XPATH, '//*[@id="radix-:rl:-trigger-rating-table"]')
    button.click()
    last_play_time = browser.find_element(By.XPATH, '//*[@id="radix-:rl:-content-rating-table"]/div/div/div/div[5]/table/tbody/tr[1]/td[1]').text
    [play_year, play_month] = last_play_time.split('-')
    if int(play_month) > datetime.datetime.now().month or int(play_year) > datetime.datetime.now().year:
        past_rating = browser.find_element(By.XPATH, '//*[@id="radix-:rl:-content-rating-table"]/div/div/div/div[5]/table/tbody/tr[3]/td[2]').text
    else:
        past_rating = browser.find_element(By.XPATH, '//*[@id="radix-:rl:-content-rating-table"]/div/div/div/div[5]/table/tbody/tr[2]/td[2]').text
    df_data['Full Name'].append(first_name + ' ' + last_name)
    df_data['Regular Rating'].append(int(reg_rating))
    df_data['Blitz Rating'].append(blitz_rating)
    df_data['Improvement'].append(int(reg_rating)-int(past_rating))

#Sample Input
id_list = "16868068 31221158 16893742 30835739".split()
for id in id_list:
    get_rating(id)
    print("Data from ID", id,"collected")

df = pd.DataFrame(df_data)

top_reg_rating_df = df.sort_values('Regular Rating', ascending=False).iloc[:, 0:2].set_index('Full Name')
print('\n')
print('Sorted by Regular Rating:')
print(top_reg_rating_df)

top_blitz_rating_df = df.sort_values('Blitz Rating', ascending=False).loc[:, ["Full Name", "Blitz Rating"]].set_index('Full Name')
print('\n')
print('Sorted by Blitz Rating:')
print(top_blitz_rating_df)

top_improvement_df = df.sort_values('Improvement', ascending=False).loc[:, ["Full Name", "Improvement"]].set_index('Full Name')
print('\n')
print('Sorted by Most Improvement:')
print(top_improvement_df)

1645
Data from ID 16868068 collected
1782
Data from ID 31221158 collected
1773
Data from ID 16893742 collected
1362
Data from ID 30835739 collected


Sorted by Regular Rating:
                      Regular Rating
Full Name                           
PRANAV SRINIVASAN               1773
VAHINI SADHU VENKATA            1744
AZIM AHMAD JULKIPLI             1710
JAMES ZHENG                     1385


Sorted by Blitz Rating:
                     Blitz Rating
Full Name                        
PRANAV SRINIVASAN            1443
VAHINI SADHU VENKATA         1257
JAMES ZHENG                  1088
AZIM AHMAD JULKIPLI          ----


Sorted by Most Improvement:
                      Improvement
Full Name                        
AZIM AHMAD JULKIPLI            65
JAMES ZHENG                    23
PRANAV SRINIVASAN              -9
VAHINI SADHU VENKATA          -29
